# RWY world life list × AviList

This notebook crosswalks a personal eBird world life list against **AviList v2025** species metadata: sunburst completion, order/family summaries, accumulation curves, and region breakdowns. The companion notebook `avilist_birds_explore.ipynb` covers AviList history, taxonomy, evolution, geography, and conservation.


In [ ]:
from __future__ import annotations

import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

try:
    from IPython.display import HTML, display
except ImportError:
    display = print
    HTML = lambda x: x


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if not (cand / "requirements.txt").exists():
            continue
        if (cand / "python" / "birds_nb.py").is_file():
            return cand
        if (cand / "birds_nb.py").is_file():
            return cand
    return p


REPO_ROOT = _repo_root()
_py = REPO_ROOT / "python"
if (_py / "birds_nb.py").is_file():
    sys.path.insert(0, str(_py))

from birds_nb import (
    CONTINENT_DISPLAY,
    family_label,
    genus_label,
    order_label,
    sp_region_label,
    sunburst_panzoom_viewport,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 130

DATA_DIR = REPO_ROOT / "data" if (REPO_ROOT / "data").is_dir() else REPO_ROOT
AVILIST_XLSX = DATA_DIR / "AviList-v2025-11Jun-extended.xlsx"
LIFELIST_CSV = DATA_DIR / "RWY_ebird_world_life_list.csv"
CACHE = DATA_DIR / ".cache_avilist.pkl.gz"
for p in (AVILIST_XLSX, LIFELIST_CSV):
    assert p.exists(), f"Missing {p}"
print(f"AviList {AVILIST_XLSX.name} ({AVILIST_XLSX.stat().st_size/1e6:.1f} MB) | list {LIFELIST_CSV.name} ({LIFELIST_CSV.stat().st_size/1e3:.1f} KB)")
# Geography choropleth: set EBIRD_API_KEY (free) — https://ebird.org/api/keygen


### Data loading and preparation

In [ ]:
from birds_nb import add_genus_common_example, continents_in, countries_in, load_avilist, regions_in

YEAR_RE = re.compile(r"(1[5-9]\d{2}|20\d{2})")

def _yr(v):
    if not isinstance(v, str):
        return np.nan
    m = YEAR_RE.search(v)
    return float(m[1]) if m else np.nan

# Load + derive all Step 1 working columns in one place.
df_all = load_avilist(AVILIST_XLSX, CACHE)
df_all["Description_year"] = df_all["Authority"].map(_yr)
df_all["Genus"] = df_all["Scientific_name"].astype(str).str.split().str[0]

iucn_order = ["LC", "NT", "VU", "EN", "CR", "EW", "EX", "DD", "NE"]
raw_iucn = df_all["IUCN_Red_List_Category"].fillna("NE").astype(str).str.strip()
raw_iucn = raw_iucn.str.replace(r"^CR.*", "CR", regex=True).where(raw_iucn.isin(iucn_order), "NE")
df_all["IUCN"] = pd.Categorical(raw_iucn, categories=iucn_order, ordered=True)

extinct_raw = df_all["Extinct_or_possibly_extinct"].astype(str).str.strip().str.lower()
df_all["is_extinct"] = extinct_raw.isin({"extinct", "possibly extinct", "yes", "true", "1"}) | df_all["IUCN"].isin(["EX", "EW"])

df_species = df_all[df_all["Taxon_rank"] == "species"].copy().reset_index(drop=True)
df_species = add_genus_common_example(df_species)
df_family = df_all[df_all["Taxon_rank"] == "family"].copy().reset_index(drop=True)
df_order = df_all[df_all["Taxon_rank"] == "order"].copy().reset_index(drop=True)

df_species["Range_continents"] = df_species["Range"].map(continents_in)
df_species["Range_regions"] = df_species["Range"].map(regions_in)
df_species["Range_countries"] = df_species["Range"].map(countries_in)
df_species["N_continents"] = df_species["Range_continents"].map(len)

parsed = (df_species["N_continents"] > 0).sum()
country_parsed = (df_species["Range_countries"].map(len) > 0).sum()
print(
    f"Loaded {len(df_all):,} rows | orders {len(df_order)} | families {len(df_family)} | "
    f"genera {df_species['Genus'].nunique():,} | species {len(df_species):,} | "
    f"continent parsed {parsed:,}/{len(df_species):,} | "
    f"country parsed {country_parsed:,}/{len(df_species):,}"
)


## 1) eBird life list integration

Now that I've imported all of the species metadata from the extended AviList, I wanted to see what I could learn about my personal life list which, at the time of writing this, sits at 1274 species. 

In [ ]:
life = pd.read_csv(LIFELIST_CSV)
life.columns = [c.strip() for c in life.columns]
life["Date"] = pd.to_datetime(life["Date"], format="%d %b %Y", errors="coerce")
life = life.rename(columns={"Scientific Name": "Scientific_name", "Common Name": "Common_name"})
cols = ["Scientific_name", "Common_name", "Location", "S/P", "Date"]
matched = df_species.merge(life[cols], on="Scientific_name", how="left", indicator=True)
m = matched["_merge"] == "both"
first_seen = matched[m].sort_values("Date").groupby("Scientific_name", as_index=False).first()[cols]
df_species_life = df_species.merge(first_seen, on="Scientific_name", how="left")
df_species_life["seen"] = df_species_life["Date"].notna()
n_seen = int(df_species_life["seen"].sum())
miss = life[~life["Scientific_name"].isin(df_species["Scientific_name"])]


As a first pass, I generated another interactive sunburst plot and colored it this time by the percentage of species seen in each family.

In [ ]:
fam_pct = (
    df_species_life.groupby(["Order", "Family", "Family_English_name"], dropna=False)
    .agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
)
fam_pct["pct_seen"] = fam_pct["n_seen"] / fam_pct["n_species"] * 100
fam_pct["Order_plot"] = fam_pct["Order"].map(order_label)
fam_pct["Family_plot"] = [family_label(f, e) for f, e in zip(fam_pct["Family"], fam_pct["Family_English_name"])]

life_sb = (
    df_species_life.groupby(["Order", "Family", "Family_English_name", "Genus", "Genus_common_example"], dropna=False)
    .agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
)
life_sb["pct_seen"] = life_sb["n_seen"] / life_sb["n_species"] * 100
life_sb["Order_plot"] = life_sb["Order"].map(order_label)
life_sb["Family_plot"] = [family_label(f, e) for f, e in zip(life_sb["Family"], life_sb["Family_English_name"])]
life_sb["Genus_plot"] = [genus_label(g, h) for g, h in zip(life_sb["Genus"], life_sb["Genus_common_example"])]

fig = px.sunburst(
    life_sb,
    path=["Order_plot", "Family_plot", "Genus_plot"],
    values="n_species",
    color="pct_seen",
    color_continuous_scale="YlOrRd",
    range_color=(0, 100),
    title="Life list % seen (Order→Family→Genus)",
    width=900,
    height=900,
)

trace = fig.data[0]
id_to_parent = dict(zip(trace.ids, trace.parents))
id_to_label = dict(zip(trace.ids, trace.labels))

orders, families, genera = [], [], []
for node_id in trace.ids:
    parent_id = id_to_parent.get(node_id, "")
    if not parent_id:
        order_plot = id_to_label.get(node_id, "")
        family_plot = ""
        genus_plot = ""
    else:
        grandparent_id = id_to_parent.get(parent_id, "")
        if not grandparent_id:
            order_plot = id_to_label.get(parent_id, "")
            family_plot = id_to_label.get(node_id, "")
            genus_plot = ""
        else:
            order_plot = id_to_label.get(grandparent_id, "")
            family_plot = id_to_label.get(parent_id, "")
            genus_plot = id_to_label.get(node_id, "")

    orders.append(order_plot)
    families.append(family_plot)
    genera.append(genus_plot)


def _life_sb_seen_total(order_plot, family_plot, genus_plot):
    d = life_sb
    if genus_plot:
        m = (d["Order_plot"] == order_plot) & (d["Family_plot"] == family_plot) & (d["Genus_plot"] == genus_plot)
        sub = d.loc[m]
    elif family_plot:
        sub = d.loc[(d["Order_plot"] == order_plot) & (d["Family_plot"] == family_plot)]
    elif order_plot:
        sub = d.loc[d["Order_plot"] == order_plot]
    else:
        sub = d
    if sub.empty:
        return 0, 0
    return int(sub["n_seen"].sum()), int(sub["n_species"].sum())


seen_lines = []
for o, f, g in zip(orders, families, genera):
    nv, ns = _life_sb_seen_total(o, f, g)
    pct = 100.0 * nv / ns if ns else 0.0
    seen_lines.append(f"% seen: {pct:.0f}% ({nv}/{ns})")

fig.update_traces(
    customdata=list(zip(orders, families, genera, seen_lines)),
    hovertemplate=(
        "Order: %{customdata[0]}<br>"
        "Family: %{customdata[1]}<br>"
        "Genus: %{customdata[2]}<br>"
        "%{customdata[3]}"
        "<extra></extra>"
    ),
)

fig.update_layout(
    dragmode="pan",
    margin=dict(t=65, l=30, r=30, b=30),
    uirevision="sunburst-life-pct",
)
_cfg = {"scrollZoom": True, "displayModeBar": True, "doubleClick": "reset", "responsive": True}
SUNBURST_GD_ID = "sunburst-life-avilist"
fig_html = pio.to_html(fig, include_plotlyjs="cdn", full_html=False, config=_cfg, div_id=SUNBURST_GD_ID)
display(HTML(sunburst_panzoom_viewport(fig_html, SUNBURST_GD_ID, 900, 900)))


In [ ]:
ord_stats = df_species_life.groupby("Order").agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
ord_stats["pct_seen"] = ord_stats["n_seen"] / ord_stats["n_species"] * 100
ord_plot = ord_stats.sort_values("pct_seen", ascending=True)
yl = ord_plot["Order"].map(order_label)
fig, ax = plt.subplots(figsize=(11, 12))
ax.barh(yl, ord_plot["pct_seen"], color=sns.color_palette("crest", n_colors=len(ord_plot)))
for i, (p, n, s) in enumerate(zip(ord_plot["pct_seen"], ord_plot["n_species"], ord_plot["n_seen"])):
    ax.text(p + 0.5, i, f"{s}/{n}", va="center", fontsize=7)
ax.set(xlabel="% order seen", xlim=(0, 100), title="Completion by order")
plt.tight_layout()
plt.show()


In [ ]:
untouched = ord_stats.query("n_seen == 0").sort_values("n_species", ascending=False).copy()
untouched.insert(0, "Order (readable)", untouched["Order"].map(order_label))
print(f"No species yet in {len(untouched)} orders:")
untouched


In [ ]:
min_size = 10
completion = fam_pct.query("n_species >= @min_size").sort_values("pct_seen", ascending=False).head(25).iloc[::-1]
labels = [f"{family_label(r.Family, r.Family_English_name)}  ({int(r.n_seen)}/{int(r.n_species)})" for r in completion.itertuples(index=False)]
fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(labels, completion["pct_seen"], color=sns.color_palette("crest", n_colors=len(completion)))
ax.set(xlabel="% family seen", xlim=(0, 100), title=f"Most complete families (≥{min_size} sp.)")
for b, pct in zip(bars, completion["pct_seen"]):
    ax.text(pct + 0.5, b.get_y() + b.get_height() / 2, f"{pct:.0f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
accum = df_species_life.dropna(subset=["Date"]).sort_values("Date").assign(cum=lambda d: range(1, len(d) + 1))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(accum["Date"], accum["cum"], color="#3d7ea0", lw=1.5)
ax.fill_between(accum["Date"], accum["cum"], alpha=0.15, color="#3d7ea0")
ax.set(xlabel="date", ylabel="cumulative species", title=f"Species accumulation ({int(accum['cum'].iloc[-1])})")
plt.tight_layout()
plt.show()


In [ ]:
by_region = df_species_life.dropna(subset=["Date"]).groupby("S/P").agg(n=("Scientific_name", "size"), n_families=("Family", "nunique")).sort_values("n", ascending=False).reset_index()
print(f"{len(by_region)} eBird regions")
top = by_region.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top["S/P"].map(sp_region_label), top["n"], color="#3d7ea0")
for i, (n, f) in enumerate(zip(top["n"], top["n_families"])):
    ax.text(n + 1, i, f"{n} sp / {f} fam", va="center", fontsize=8)
ax.set(xlabel="species (first seen in region)", title="Top 15 regions (eBird S/P codes)")
plt.tight_layout()
plt.show()


In [ ]:
rf = df_species_life.dropna(subset=["Date"]).groupby(["S/P", "Family"]).size().unstack(fill_value=0)
top_regions = by_region["S/P"].head(15).tolist()
top_fams = df_species_life.dropna(subset=["Date"])["Family"].value_counts().head(25).index.tolist()
rf_sub = rf.reindex(index=top_regions, columns=top_fams).fillna(0)
fe = df_family.set_index("Scientific_name")["Family_English_name"].to_dict()
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(rf_sub, cmap="YlGnBu", annot=True, fmt=".0f", cbar_kws={"label": "species"}, lw=0.3, ax=ax, xticklabels=[family_label(f, fe.get(f, "")) for f in rf_sub.columns], yticklabels=[sp_region_label(r) for r in rf_sub.index])
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_title("Region × family lifers")
plt.tight_layout()
plt.show()


In [ ]:
dom = (
    df_species_life.dropna(subset=["Date"]).groupby(["S/P", "Family"]).size().reset_index(name="n")
    .sort_values(["S/P", "n"], ascending=[True, False]).drop_duplicates("S/P").sort_values("n", ascending=False).head(25).reset_index(drop=True)
)
dom
